## 03e_dashboard_performance_tables — RCM Intelligence Platform
### Purpose
Builds pre-aggregated dashboard tables to eliminate query latency in Streamlit app.
All heavy aggregations are materialized here and refreshed daily.
This reduces dashboard load time from 5-10 seconds to <1 second.

### Tables Built
| Table | Description | Used By |
|-------|-------------|----------|
| dashboard_executive_kpis | Single-row national KPI summary | Executive Overview page |
| dashboard_state_summary | State-level aggregations (top 50) | Executive Overview charts |
| dashboard_drg_leakage | Top 20 DRGs by revenue gap | Executive Overview + Hospital 360 |
| dashboard_ar_aging | AR aging bucket summary (4 rows) | Executive Overview chart |
| dashboard_grade_distribution | Hospital 360 grade counts | Executive Overview pie chart |
| dashboard_national_benchmarks | Pre-computed national percentile ranks | Hospital 360 KPI cards |

### Performance Impact
- Before: Each dashboard page runs 5-10 aggregation queries (3-8 sec load time)
- After: Direct SELECT from pre-aggregated tables (<500ms load time)
- Data freshness: Daily refresh via scheduled job

### Inputs
- rcm_platform.rcm_gold.fact_claims
- rcm_platform.rcm_gold.hospital_scorecard
- rcm_platform.rcm_gold.hospital_360_scorecard
- rcm_platform.rcm_gold.combined_hospital_scorecard
- rcm_platform.rcm_gold.kpi_by_state
- rcm_platform.rcm_gold.kpi_by_drg
- rcm_platform.rcm_silver.hospital_quality_scores
- rcm_platform.rcm_silver.hospital_hcahps_scores

### Outputs
- 6 optimized dashboard tables in rcm_platform.rcm_gold

| Field | Value |
|-------|-------|
| Author | Mayank Joshi |
| Layer | Gold (Dashboard Optimization) |
| Run order | 5 — after 03a-03d Gold notebooks |
| Depends on | 00_config, 00_utils, all Gold tables |
| Schedule | Daily at 6 AM UTC |

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_config

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_utils

In [0]:
# ============================================================
# BATCH METADATA
# ============================================================

import uuid
from datetime import datetime, timezone
from pyspark.sql.functions import (
    col, lit, when, round as spark_round, expr, sum as spark_sum, avg,
    count, countDistinct, max, min, percent_rank,
    row_number, rank, desc, asc
)
from pyspark.sql.window import Window

BATCH_ID        = str(uuid.uuid4())
BATCH_DATE      = datetime.now(timezone.utc).strftime("%Y-%m-%d")
BATCH_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
NOTEBOOK_NAME   = "03e_dashboard_performance_tables"

print(f"Batch ID        : {BATCH_ID}")
print(f"Batch date      : {BATCH_DATE}")
print(f"Batch timestamp : {BATCH_TIMESTAMP}")

# Read source tables
df_fact = spark.table(TBL_FACT_CLAIMS)
print(f"\nFact table rows : {df_fact.count():>8,}")

In [0]:
# ============================================================
# TABLE 1 — DASHBOARD_EXECUTIVE_KPIS
# Single-row summary for Executive Overview page
# Replaces ~5 separate aggregation queries with 1 SELECT
# ============================================================

print("=" * 70)
print("  TABLE 1: dashboard_executive_kpis")
print("=" * 70)

# Main aggregation from fact table - using deduped provider_id+drg_code
df_exec_base = spark.sql("""
    SELECT DISTINCT
        provider_id,
        provider_state,
        drg_code,
        total_discharges,
        charge_to_payment_ratio,
        total_revenue_gap,
        underpayment_flag
    FROM rcm_platform.rcm_gold.fact_claims
""")

df_exec_kpis = df_exec_base.select(
    countDistinct("provider_id").alias("total_hospitals"),
    countDistinct("provider_state").alias("states_covered"),
    spark_round((spark_sum("total_discharges") / 1000000), 2).alias("total_discharges_millions"),
    spark_round(avg("charge_to_payment_ratio"), 2).alias("avg_ctp_ratio"),
    spark_round((spark_sum(when(col("underpayment_flag") == 1, 1).otherwise(0)) / count("*") * 100), 2).alias("underpayment_rate_pct"),
    spark_round((spark_sum("total_revenue_gap") / 1000000000), 2).alias("inpatient_gap_billions")
)

# Add combined gap from hospital scorecard
df_combined_gap = spark.sql("""
    SELECT ROUND(SUM(combined_revenue_gap_billions), 2) AS combined_gap_billions
    FROM rcm_platform.rcm_gold.combined_hospital_scorecard
""")

# Cross join to add combined gap (both are single-row DataFrames)
df_exec_kpis = df_exec_kpis.crossJoin(df_combined_gap)

# Add refresh metadata
df_exec_kpis = df_exec_kpis.withColumn("last_refreshed", lit(BATCH_TIMESTAMP).cast("timestamp"))

# Write to Gold
df_exec_kpis.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_DB}.dashboard_executive_kpis")

spark.sql(f"""
    COMMENT ON TABLE {GOLD_DB}.dashboard_executive_kpis
    IS 'Pre-aggregated Executive Overview KPIs — single-row summary for dashboard. Refreshed daily.'
""")

print(f"✓ dashboard_executive_kpis created (1 row)")
df_exec_kpis.show(truncate=False)

In [0]:
# ============================================================
# TABLE 2 — DASHBOARD_STATE_SUMMARY
# State-level metrics for geographic visualizations
# Pre-computes both inpatient and outpatient state aggregations
# ============================================================

print("\n" + "=" * 70)
print("  TABLE 2: dashboard_state_summary")
print("=" * 70)

# Inpatient state metrics from fact_claims
df_state_inpatient = spark.sql("""
    SELECT
        provider_state,
        ROUND(SUM(total_revenue_gap) / 1e9, 2) AS inpatient_revenue_gap_billions
    FROM (
        SELECT DISTINCT 
            provider_id, 
            provider_state, 
            drg_code, 
            total_revenue_gap
        FROM rcm_platform.rcm_gold.fact_claims
    )
    GROUP BY provider_state
""")

# Outpatient state metrics
df_state_outpatient = spark.sql("""
    SELECT
        provider_state,
        ROUND(total_revenue_gap_billions, 2) AS outpatient_revenue_gap_billions
    FROM rcm_platform.rcm_gold.outpatient_kpi_by_state
""")

# Join inpatient and outpatient metrics
df_state_summary = df_state_inpatient.join(
    df_state_outpatient,
    "provider_state",
    "left"
).withColumn(
    "combined_revenue_gap_billions",
    spark_round(col("inpatient_revenue_gap_billions") + coalesce(col("outpatient_revenue_gap_billions"), lit(0)), 2)
)

# Add hospital count and other metrics from kpi_by_state
df_state_kpis = spark.sql("""
    SELECT
        provider_state,
        hospital_count,
        total_discharges,
        ROUND(avg_ctp_ratio, 2) AS avg_ctp_ratio,
        ROUND(avg_revenue_gap_pct, 2) AS avg_revenue_gap_pct,
        ROUND(underpayment_rate_pct, 2) AS underpayment_rate_pct
    FROM rcm_platform.rcm_gold.kpi_by_state
""")

df_state_summary = df_state_summary.join(
    df_state_kpis,
    "provider_state",
    "left"
).orderBy(desc("combined_revenue_gap_billions"))

# Add refresh metadata
df_state_summary = df_state_summary.withColumn("last_refreshed", lit(BATCH_TIMESTAMP).cast("timestamp"))

# Write to Gold
df_state_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_DB}.dashboard_state_summary")

spark.sql(f"""
    COMMENT ON TABLE {GOLD_DB}.dashboard_state_summary
    IS 'Pre-aggregated state-level metrics for Executive Overview charts. Top 50 states by revenue gap.'
""")

# Optimize for state lookups
spark.sql(f"OPTIMIZE {GOLD_DB}.dashboard_state_summary ZORDER BY (provider_state)")

print(f"✓ dashboard_state_summary created ({df_state_summary.count()} states)")
df_state_summary.orderBy(desc("combined_revenue_gap_billions")).show(10, truncate=False)

In [0]:
# ============================================================
# TABLE 3 — DASHBOARD_DRG_LEAKAGE
# Top 20 DRGs by revenue gap
# Used in Executive Overview and Hospital 360 pages
# ============================================================

print("\n" + "=" * 70)
print("  TABLE 3: dashboard_drg_leakage")
print("=" * 70)

df_drg_leakage = spark.sql("""
    SELECT
        drg_code,
        drg_description,
        SUM(total_discharges) AS total_discharges,
        ROUND(SUM(total_revenue_gap) / 1e9, 2) AS revenue_gap_billions,
        ROUND(AVG(charge_to_payment_ratio), 2) AS avg_ctp_ratio,
        ROUND(AVG(revenue_gap_pct), 1) AS avg_revenue_gap_pct,
        COUNT(DISTINCT provider_id) AS hospital_count
    FROM (
        SELECT DISTINCT 
            provider_id,
            drg_code,
            drg_description,
            total_discharges,
            total_revenue_gap,
            charge_to_payment_ratio,
            revenue_gap_pct
        FROM rcm_platform.rcm_gold.fact_claims
    )
    GROUP BY drg_code, drg_description
    ORDER BY revenue_gap_billions DESC
    LIMIT 20
""")

# Add rank column
window_rank = Window.orderBy(desc("revenue_gap_billions"))
df_drg_leakage = df_drg_leakage.withColumn("rank", row_number().over(window_rank))

# Add refresh metadata
df_drg_leakage = df_drg_leakage.withColumn("last_refreshed", lit(BATCH_TIMESTAMP).cast("timestamp"))

# Write to Gold
df_drg_leakage.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_DB}.dashboard_drg_leakage")

spark.sql(f"""
    COMMENT ON TABLE {GOLD_DB}.dashboard_drg_leakage
    IS 'Top 20 DRGs by revenue gap for Executive Overview leakage chart. Refreshed daily.'
""")

print(f"✓ dashboard_drg_leakage created (20 rows)")
df_drg_leakage.select("rank", "drg_code", "drg_description", "revenue_gap_billions", "total_discharges").show(10, truncate=False)

In [0]:
# ============================================================
# TABLE 4 — DASHBOARD_AR_AGING
# AR aging buckets aggregated to 4 rows
# Replaces complex aggregation with simple SELECT
# ============================================================

print("\n" + "=" * 70)
print("  TABLE 4: dashboard_ar_aging")
print("=" * 70)

df_ar_aging = spark.sql("""
    SELECT
        ar_aging_bucket,
        ROUND(SUM(revenue_at_risk) / 1e9, 2) AS revenue_at_risk_billions,
        CASE ar_aging_bucket
            WHEN '0-30 days' THEN 1
            WHEN '31-60 days' THEN 2
            WHEN '61-90 days' THEN 3
            WHEN '90+ days' THEN 4
        END AS bucket_order
    FROM rcm_platform.rcm_gold.ar_aging_buckets
    GROUP BY ar_aging_bucket
    ORDER BY bucket_order
""")

# Add refresh metadata
df_ar_aging = df_ar_aging.withColumn("last_refreshed", lit(BATCH_TIMESTAMP).cast("timestamp"))

# Write to Gold
df_ar_aging.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_DB}.dashboard_ar_aging")

spark.sql(f"""
    COMMENT ON TABLE {GOLD_DB}.dashboard_ar_aging
    IS 'AR aging buckets aggregated for Executive Overview chart. 4 rows total.'
""")

print(f"✓ dashboard_ar_aging created (4 rows)")
df_ar_aging.select("ar_aging_bucket", "revenue_at_risk_billions").show(truncate=False)

In [0]:
# ============================================================
# TABLE 5 — DASHBOARD_GRADE_DISTRIBUTION
# Hospital 360 grade counts for pie chart
# Simple 5-row table (A, B, C, D, F grades)
# ============================================================

print("\n" + "=" * 70)
print("  TABLE 5: dashboard_grade_distribution")
print("=" * 70)

df_grade_dist = spark.sql("""
    SELECT
        hospital_360_grade,
        COUNT(*) AS hospital_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS percentage
    FROM rcm_platform.rcm_gold.hospital_360_scorecard
    WHERE hospital_360_grade != 'Insufficient Data'
    GROUP BY hospital_360_grade
    ORDER BY hospital_count DESC
""")

# Add grade order for consistent sorting
df_grade_dist = df_grade_dist.withColumn(
    "grade_order",
    when(col("hospital_360_grade") == "A — Excellent", 1)
    .when(col("hospital_360_grade") == "B — Good", 2)
    .when(col("hospital_360_grade") == "C — Average", 3)
    .when(col("hospital_360_grade") == "D — Below Average", 4)
    .when(col("hospital_360_grade") == "F — Critical", 5)
    .otherwise(6)
).orderBy("grade_order")

# Add refresh metadata
df_grade_dist = df_grade_dist.withColumn("last_refreshed", lit(BATCH_TIMESTAMP).cast("timestamp"))

# Write to Gold
df_grade_dist.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_DB}.dashboard_grade_distribution")

spark.sql(f"""
    COMMENT ON TABLE {GOLD_DB}.dashboard_grade_distribution
    IS 'Hospital 360 grade distribution for pie chart on Executive Overview. ~5 rows.'
""")

print(f"✓ dashboard_grade_distribution created ({df_grade_dist.count()} grades)")
df_grade_dist.select("hospital_360_grade", "hospital_count", "percentage").show(truncate=False)

In [0]:
# ============================================================
# TABLE 6 — DASHBOARD_NATIONAL_BENCHMARKS
# Pre-compute national percentile ranks for all hospitals
# This is the KEY performance optimization - percentiles are expensive
# Adds national percentile rank to every hospital KPI card
# ============================================================

print("\n" + "=" * 70)
print("  TABLE 6: dashboard_national_benchmarks (CRITICAL OPTIMIZATION)")
print("=" * 70)

# Read combined hospital scorecard and join with hospital_360 for the 360 score
df_combined = spark.table("rcm_platform.rcm_gold.combined_hospital_scorecard")
df_360 = spark.table("rcm_platform.rcm_gold.hospital_360_scorecard").select(
    "provider_id",
    "hospital_360_score",
    "avg_excess_readmission_ratio"
)

df_hospitals = df_combined.join(df_360, "provider_id", "left")

# Define window for percentile ranking
window_all = Window.orderBy(col("provider_id"))

# Calculate percentile rank for each key metric
# Lower CTP = better, so use ASC for CTP
# Lower underpayment = better, so use ASC
# Higher 360 score = better, so use DESC
# Lower denial risk = better, so use ASC

df_benchmarks = df_hospitals.select(
    col("provider_id"),
    col("provider_name"),
    col("provider_state"),
    
    # CTP Ratio percentile (lower is better)
    spark_round(
        (percent_rank().over(Window.orderBy(col("inpatient_avg_ctp"))) * 100), 1
    ).alias("ctp_percentile"),
    col("inpatient_avg_ctp"),
    
    # Underpayment Rate percentile (lower is better)
    spark_round(
        (percent_rank().over(Window.orderBy(col("inpatient_underpayment_rate"))) * 100), 1
    ).alias("underpayment_percentile"),
    col("inpatient_underpayment_rate"),
    
    # Hospital 360 Score percentile (higher is better)
    spark_round(
        (percent_rank().over(Window.orderBy(desc("hospital_360_score"))) * 100), 1
    ).alias("hospital_360_percentile"),
    col("hospital_360_score"),
    
    # RCM Health Score percentile (higher is better)
    spark_round(
        (percent_rank().over(Window.orderBy(desc("rcm_health_score"))) * 100), 1
    ).alias("rcm_health_percentile"),
    col("rcm_health_score"),
    
    # Denial Risk percentile (lower is better) - using readmission ratio as proxy
    spark_round(
        (percent_rank().over(Window.orderBy(col("avg_excess_readmission_ratio"))) * 100), 1
    ).alias("denial_risk_percentile"),
    col("avg_excess_readmission_ratio").alias("avg_denial_risk_score"),
    
    # Revenue Gap percentile (lower is better)
    spark_round(
        (percent_rank().over(Window.orderBy(col("combined_revenue_gap_billions"))) * 100), 1
    ).alias("revenue_gap_percentile"),
    col("combined_revenue_gap_billions")
)

# Add refresh metadata
df_benchmarks = df_benchmarks.withColumn("last_refreshed", lit(BATCH_TIMESTAMP).cast("timestamp"))

# Write to Gold
df_benchmarks.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_DB}.dashboard_national_benchmarks")

spark.sql(f"""
    COMMENT ON TABLE {GOLD_DB}.dashboard_national_benchmarks
    IS 'Pre-computed national percentile ranks for all hospitals on key metrics. CRITICAL for dashboard performance.'
""")

# Optimize for hospital lookups
spark.sql(f"OPTIMIZE {GOLD_DB}.dashboard_national_benchmarks ZORDER BY (provider_id, provider_state)")

print(f"✓ dashboard_national_benchmarks created ({df_benchmarks.count()} hospitals)")
print("\nSample percentile ranks:")
df_benchmarks.select(
    "provider_name", 
    "provider_state",
    "ctp_percentile", 
    "hospital_360_percentile",
    "denial_risk_percentile"
).orderBy(desc("hospital_360_percentile")).show(10, truncate=False)

In [0]:
# ============================================================
# VERIFICATION — ROW COUNTS FOR ALL DASHBOARD TABLES
# ============================================================

print("\n" + "=" * 70)
print("  VERIFICATION: Dashboard Table Row Counts")
print("=" * 70)

table_checks = [
    "dashboard_executive_kpis",
    "dashboard_state_summary",
    "dashboard_drg_leakage",
    "dashboard_ar_aging",
    "dashboard_grade_distribution",
    "dashboard_national_benchmarks"
]

for table_name in table_checks:
    full_table = f"{GOLD_DB}.{table_name}"
    row_count = spark.table(full_table).count()
    print(f"  {table_name:<35} : {row_count:>8,} rows")

print("\n✓ All dashboard tables created successfully")

In [0]:
# ============================================================
# AUDIT LOGGING
# ============================================================

from pyspark.sql.functions import lit

audit_records = []

for table_name in table_checks:
    full_table = f"{GOLD_DB}.{table_name}"
    row_count = spark.table(full_table).count()
    
    audit_records.append((
        BATCH_ID,
        BATCH_DATE,
        BATCH_TIMESTAMP,
        NOTEBOOK_NAME,
        table_name,
        row_count,
        "SUCCESS"
    ))

audit_schema = ["batch_id", "batch_date", "batch_timestamp", "notebook_name", 
                "table_name", "row_count", "status"]

df_audit = spark.createDataFrame(audit_records, audit_schema)

df_audit.write \
    .mode("append") \
    .saveAsTable(f"{GOLD_DB}.dashboard_audit_log")

print(f"\n✓ Audit records written to {GOLD_DB}.dashboard_audit_log")
print(f"  Batch ID: {BATCH_ID}")

In [0]:
# ============================================================
# PIPELINE SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("  DASHBOARD PERFORMANCE OPTIMIZATION — SUMMARY")
print("=" * 70)
print(f"\n  Batch ID        : {BATCH_ID}")
print(f"  Batch timestamp : {BATCH_TIMESTAMP}")
print(f"  Notebook        : {NOTEBOOK_NAME}")
print("\n  Tables built:")
print("  " + "-" * 66)

total_rows = 0
for table_name in table_checks:
    full_table = f"{GOLD_DB}.{table_name}"
    row_count = spark.table(full_table).count()
    total_rows += row_count
    print(f"  ✓ {table_name:<35} : {row_count:>8,} rows")

print("  " + "-" * 66)
print(f"  Total rows across all dashboard tables  : {total_rows:>8,}")
print("\n  Performance impact:")
print("    Before : 5-10 sec dashboard load (on-the-fly aggregations)")
print("    After  : <1 sec dashboard load (pre-aggregated tables)")
print("\n  Data freshness : Daily refresh via scheduled job")
print("\n" + "=" * 70)
print("  ✓ PIPELINE COMPLETE")
print("=" * 70)